# 02 · Baseline SFT (Supervised Fine-Tuning)

Quick-win baseline: fine-tune a small model with SFT before GRPO.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

MODEL_NAME = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 2048
BATCH_SIZE = 2
LR = 2e-4
NUM_EPOCHS = 1

print('Config loaded')

## 1. Load model with Unsloth (4-bit)

In [ ]:
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                         'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
    )
    print('Unsloth model loaded')
except ImportError:
    print('unsloth not installed – skipping model load (demo mode)')
    model = None
    tokenizer = None

## 2. Prepare dataset

In [ ]:
import json
from datasets import Dataset
from utils import build_prompt

qa_path = REPO_ROOT / 'data' / 'synthetic' / 'synthetic_qa_pairs.json'
schema_path = REPO_ROOT / 'data' / 'synthetic' / 'enterprise_schemas.json'

with open(qa_path) as f:
    qa_pairs = json.load(f)
with open(schema_path) as f:
    schemas = json.load(f)

rows = []
for ex in qa_pairs:
    schema = schemas.get(ex['db_id'], {})
    prompt = build_prompt(ex['question'], schema)
    rows.append({'text': prompt + ex['sql'] + '\n'})

dataset = Dataset.from_list(rows)
print(f'Dataset size: {len(dataset)}')
print(dataset[0]['text'][:300])

## 3. SFT Training

In [ ]:
if model is not None and tokenizer is not None:
    from trl import SFTTrainer
    from transformers import TrainingArguments

    training_args = TrainingArguments(
        output_dir='/tmp/sft_baseline',
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=LR,
        bf16=True,
        logging_steps=5,
        save_strategy='no',
        report_to='none',
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        args=training_args,
    )
    trainer.train()
    print('SFT training complete')
else:
    print('Skipping training (model not loaded)')

## 4. Quick inference test

In [ ]:
if model is not None and tokenizer is not None:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

    test_question = 'What is the total revenue for orders shipped in January 1995?'
    test_schema = schemas.get('tpch', {})
    prompt = build_prompt(test_question, test_schema)

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(decoded[len(prompt):])
else:
    print('Skipping inference (model not loaded)')